# rearrange-as-sequential-layer composite — cx12: Rearrange-as-layer composes inside a nested Sequential classifier head

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `module-composition`, `rearrange-as-sequential-layer`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "rearrange-as-sequential-layer"
DD_ATOM_IDS = ["module-composition", "rearrange-as-sequential-layer"]
DD_SUBTOPICS = ["PyTorch: Module composition", "Einops: Rearrange as nn.Sequential layer"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's discriminator (and almost every conv classifier) ends with a `(B, C, H, W) -> (B, num_classes)` head. The head IS a sub-Sequential composed inside the outer model:

```python
head = nn.Sequential(
    Rearrange('b c h w -> b (c h w)'),       # rearrange-as-sequential-layer (flatten).
    nn.Linear(C * H * W, num_classes),
)
```

Composing the head INSIDE the outer feature extractor (the `module-composition` atom) yields a single Sequential-of-Sequentials with NO custom forward:

```python
classifier = nn.Sequential(
    feature_extractor,    # outer Sequential of conv blocks.
    head,                 # inner Sequential of Rearrange + Linear.
)
```

**Why both atoms together.** `Rearrange` is what makes the head a `Sequential` — without it you'd need `forward()` to call `.flatten(1)`. `module-composition` is what makes the head a child of the outer classifier — without it you'd need a custom forward to call `self.features(x)` then `self.head(...)`. Combine them and the whole classifier is DECLARATIVE.

### Composite Exercise — Rearrange-as-layer composes inside a nested Sequential classifier head

**Atoms exercised together**: `module-composition`, `rearrange-as-sequential-layer`

Implement `cx12_make_classifier(in_channels, num_classes)` — return an `nn.Sequential` with TWO children: a feature extractor and a classifier head.

Layer composition:
```
outer = nn.Sequential(
    feature_extractor,   # child 0: nn.Sequential of conv blocks.
    head,                # child 1: nn.Sequential of Rearrange + Linear.
)
```

Where:
1. `feature_extractor = nn.Sequential(`
       `nn.Conv2d(in_channels, 8, kernel_size=3, padding=1, stride=2), nn.ReLU(),`
       `nn.Conv2d(8, 16, kernel_size=3, padding=1, stride=2), nn.ReLU(),`
   `)`  # (B, in_channels, 8, 8) -> (B, 16, 2, 2).
2. `head = nn.Sequential(`
       `Rearrange('b c h w -> b (c h w)'),`
       `nn.Linear(16 * 2 * 2, num_classes),`
   `)`  # (B, 16, 2, 2) -> (B, num_classes).
3. Return `nn.Sequential(feature_extractor, head)`.

Input shape: `(B, in_channels, 8, 8)`. Output shape: `(B, num_classes)`.

Test checks:
- Outer is `nn.Sequential` with EXACTLY 2 children, both themselves `nn.Sequential`.
- Inner head's FIRST layer is a `Rearrange`.
- Inner head's SECOND layer is `nn.Linear(64, num_classes)` (since 16*2*2 = 64).
- End-to-end shape parity: `(B, in_channels, 8, 8) -> (B, num_classes)`.
- Parameters from BOTH child Sequentials are collected by `outer.parameters()`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx12_make_classifier(in_channels: int, num_classes: int) -> 'nn.Sequential':
    """Return outer nn.Sequential of (feature_extractor, head) where head uses Rearrange."""
    raise NotImplementedError

def _test_cx12():
    model = cx12_make_classifier(in_channels=3, num_classes=10)
    assert isinstance(model, nn.Sequential), f'outer must be nn.Sequential; got {type(model).__name__}'

    children = list(model.children())
    assert len(children) == 2, f'outer should have exactly 2 children; got {len(children)}'
    feature_extractor, head = children
    assert isinstance(feature_extractor, nn.Sequential), (
        f'child 0 (feature_extractor) must be Sequential; got {type(feature_extractor).__name__}'
    )
    assert isinstance(head, nn.Sequential), (
        f'child 1 (head) must be Sequential; got {type(head).__name__}'
    )

    # Case A: feature_extractor structure — 4 layers (2x conv + relu).
    fe_layers = list(feature_extractor.children())
    assert len(fe_layers) == 4, f'feature_extractor should have 4 layers; got {len(fe_layers)}'
    assert isinstance(fe_layers[0], nn.Conv2d) and fe_layers[0].in_channels == 3
    assert isinstance(fe_layers[2], nn.Conv2d) and fe_layers[2].out_channels == 16

    # Case B: head structure — Rearrange + Linear.
    head_layers = list(head.children())
    assert len(head_layers) == 2, f'head should have 2 layers; got {len(head_layers)}'
    assert isinstance(head_layers[0], Rearrange), (
        f'head[0] must be einops Rearrange layer; got {type(head_layers[0]).__name__}'
    )
    assert isinstance(head_layers[1], nn.Linear), (
        f'head[1] must be Linear; got {type(head_layers[1]).__name__}'
    )
    assert head_layers[1].in_features == 16 * 2 * 2, (
        f'head Linear in_features should be 16*2*2=64; got {head_layers[1].in_features}'
    )
    assert head_layers[1].out_features == 10

    # Case C: end-to-end shape.
    for B in (1, 4):
        x = t.randn(B, 3, 8, 8)
        out = model(x)
        assert out.shape == (B, 10), f'expected (B, num_classes)=({B}, 10); got {tuple(out.shape)}'

    # Case D: outer .parameters() recurses through BOTH children.
    all_params = list(model.parameters())
    fe_params = list(feature_extractor.parameters())
    head_params = list(head.parameters())
    assert len(all_params) == len(fe_params) + len(head_params), (
        'outer .parameters() must collect from BOTH child Sequentials transitively'
    )
    # feature_extractor has 2 Conv2d (weight + bias each) = 4 params; head has 1 Linear = 2.
    assert len(all_params) == 6, f'expected 6 total params (4 fe + 2 head); got {len(all_params)}'

    # Case E: intermediate (after feature_extractor) shape proves the (B, 16, 2, 2) handoff.
    x = t.randn(2, 3, 8, 8)
    feat = feature_extractor(x)
    assert feat.shape == (2, 16, 2, 2), (
        f'feature_extractor output should be (2, 16, 2, 2); got {tuple(feat.shape)}'
    )
    _dd_passed.add('cx12')

_test_cx12()

<details><summary>Show solution — cx12</summary>

```python
def cx12_make_classifier(in_channels: int, num_classes: int):
    # Atom A (module-composition): inner feature extractor as one child Sequential.
    feature_extractor = nn.Sequential(
        nn.Conv2d(in_channels, 8, kernel_size=3, padding=1, stride=2),
        nn.ReLU(),
        nn.Conv2d(8, 16, kernel_size=3, padding=1, stride=2),
        nn.ReLU(),
    )
    # Atom B (rearrange-as-sequential-layer): Rearrange + Linear as a Sequential head.
    head = nn.Sequential(
        Rearrange('b c h w -> b (c h w)'),
        nn.Linear(16 * 2 * 2, num_classes),
    )
    # Atom A (module-composition): wrap them in an OUTER Sequential.
    return nn.Sequential(feature_extractor, head)
```

Sequential-of-Sequentials is fully legal and registers everything transitively — `outer.parameters()` walks into the nested Sequentials and finds every learnable tensor. The whole classifier is just a tree of Sequentials with a Rearrange leaf doing the flatten — no custom forward needed anywhere. This is the cleanest version of the 'conv encoder + flatten + linear head' pattern.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx12'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx12',
        'subtopics': ["PyTorch: Module composition", "Einops: Rearrange as nn.Sequential layer"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()